# 02 — Data Quality Walkthrough

This notebook shows how to use `validation/validate_dataset.py` programmatically and how to build simple quality reports on top of it.

The validator checks required fields, duplicate IDs, controlled vocabularies, numeric ranges, country codes, collection months, and overall completeness against `validation/quality_rules.json`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "validation").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "validation"))

from validate_dataset import validate_file

DATASETS = REPO_ROOT / "synthetic-data" / "datasets"

## Validate a clean dataset

In [ ]:
report = validate_file(DATASETS / "survey_responses_1000.csv")
report.to_dict()

## See how errors are reported

Deliberately corrupt a few rows, then validate. This is the quickest way to understand the error format before building tooling on top of it.

In [ ]:
df = pd.read_csv(DATASETS / "sample_50.csv")
df.loc[0, "pain_score"] = 99            # out of range
df.loc[1, "flow_heaviness"] = "extreme"  # not in controlled vocabulary
df.loc[2, "record_id"] = df.loc[3, "record_id"]  # duplicate ID
df.loc[4, "collection_month"] = "2026-13"  # invalid month

broken_path = Path("_broken_sample.csv")
df.to_csv(broken_path, index=False)

broken_report = validate_file(broken_path)
print(f"valid: {broken_report.valid}, errors: {len(broken_report.errors)}")
pd.DataFrame([e.__dict__ for e in broken_report.errors])

In [ ]:
broken_path.unlink(missing_ok=True)

## Per-field missingness report

The validator's completeness score covers required fields. Optional fields (`region`, `reported_symptoms`) can still be missing — the dataset below was generated with `--missingness 0.15` to simulate that.

In [ ]:
sparse = pd.read_csv(DATASETS / "survey_responses_missingness_500.csv")
missing_rates = sparse.isna().mean().sort_values(ascending=False)
missing_rates[missing_rates > 0].plot.barh(title="Missingness by field (synthetic)")

## Build on this

Open contribution ideas (see `issues/issue_backlog.md`):

- Cross-field consistency checks (e.g. `period_duration_days` < `cycle_length_days`)
- Missingness pattern reports (which fields go missing together?)
- A quality dashboard page that visualizes validation reports
- Schema-driven validation that reads `schema/survey_schema.json` directly